In [2]:
from sage.schemes.elliptic_curves.hom import EllipticCurveHom

# Ancillary Functions

def frob_conj_rational_function(f, p):
    R = f.parent()
    num = f.numerator().map_coefficients(lambda c: c^p)
    den = f.denominator().map_coefficients(lambda c: c^p)
    return R(num) / R(den)

def get_j_invariants(d, p, is_proper_order):
    if is_prime(p) == False:
        raise Exception("p must be prime!")
    if d != 3:
        raise Exception("d must be 3")
        
    e_p_3 = -kronecker(-d, p)
    list_of_3_e_p_structures = []
    

    RR.<x> = PolynomialRing(QQ)
    K.<u> = QuadraticField(-d*p)
    
    if (-d*p)% 4 == 1:
        Disc = -d*p            
    elif (-d*p)% 4 == 3:
        Disc = -4*d*p
    else:
        raise Exception("Error!")

    if is_proper_order == True:
        Disc = Disc * 4
        H = hilbert_class_polynomial(Disc)
        H = H.change_ring(K)
        F = H.splitting_field("c")
    else: 
        H = hilbert_class_polynomial(Disc)
        F = K.hilbert_class_field('a')
    
    H = H.change_ring(F)
    
    j_invariants_p = []
    
    for r in H.roots(multiplicities=False):
        P = F.primes_above(p)[0]
        k.<a1> = P.residue_field()
        j_red = k(r)
        # print(j_red)
        Fq = GF(k.cardinality())
        root = k.modulus().change_ring(Fq).roots(multiplicities=False)[0]
        phi = k.hom([root], Fq)
        
        j = phi(j_red)
        j_invariants_p.append(j)
        
    j_invariants_p = Set(j_invariants_p)
    return (list(j_invariants_p), Fq)
    
def give_list_of_3_e_p_structures(E, d, p, e_p_3, Fq):
    list_of_3_e_p_structures = []
    j = E.j_invariant()
    Ep = EllipticCurve(Fq, [E.a4()^p, E.a6()^p])
    A = E.a4(); B = E.a6()
    
    f = E.division_polynomial(3)
    fac = f.factor()
    degrees = [g.degree() for g, mult in fac]
    m = lcm(degrees)
    DP_field.<z2> = Fq.extension(m)
    f = f.change_ring(DP_field)
    E = E.change_ring(DP_field)
    Ep = EllipticCurve(DP_field, [E.a4()^p, E.a6()^p])
    roots_3 = f.roots(multiplicities=False)
    for r_P in roots_3:
        try:
            P = E.lift_x(r_P)
        except:
            DP_field_ext.<u1> = DP_field.extension(2)
            E = E.change_ring(DP_field_ext)
            Ep = Ep.change_ring(DP_field_ext)
            P = E.lift_x(r_P)
            
        F_map = E.isogeny(P)
        E1 = F_map.codomain()
        signs = [1, -1]
        for sign in signs:
            if j^p != E1.j_invariant():
                continue  
            tau = E1.isomorphism_to(Ep)
            psi = sign * tau * F_map
            X = psi.rational_maps()[0]
            Y = psi.rational_maps()[1]
            X_p = frob_conj_rational_function(X, p)
            Y_p = frob_conj_rational_function(Y, p)
            
            x, y = Y.parent().gens()

            if Y_p/psi.dual().rational_maps()[1] == e_p_3 and X(x=x^(p^2), y=y^(p^2)) == X^(p^2) and Y(x=x^(p^2), y=y^(p^2)) == Y^(p^2):
                list_of_3_e_p_structures.append((E, psi, P))
    
    return list_of_3_e_p_structures

In [5]:
# Main 

def give_explicit_d_e_structure(d, p, is_D_e_p_sub, e):
    e_p_3 = e
    is_D_e_p_sub_flag = is_D_e_p_sub
    if (-d*p) % 4 == 3:
        is_D_e_p_sub = False    
    elif (-d*p) % 4 != 1:
        raise Exception("Invalid inputs")
    
    list_of_3_e_p_structures = []
    
    (j_invariants_p, Fq) = get_j_invariants(d, p, is_D_e_p_sub)
    
    for j in j_invariants_p:
        E = EllipticCurve(Fq, j = j)
        ns = Fq.multiplicative_generator()
        E_twist = E.quadratic_twist(ns)
        list_of_3_e_p_structures += give_list_of_3_e_p_structures(E, d, p, e_p_3, Fq)
        list_of_3_e_p_structures += give_list_of_3_e_p_structures(E_twist, d, p, e_p_3, Fq)
    # print(len(list_of_3_e_p_structures))
    if (-d*p) % 4 == 3:
        if is_D_e_p_sub_flag == True:
            return []
        else:
            return list_of_3_e_p_structures
    elif (-d*p) % 4 == 1:
        list_of_3_e_p_structures_sub = []
        list_of_3_e_p_structures_max = []
        for l_item in list_of_3_e_p_structures:
            flag = False
            E_item = l_item[0]
            X = (l_item[1].rational_maps()[0])^p
            Y = (l_item[1].rational_maps()[1])^p
            div_2_points = [E_item(r, 0) for r in E_item.division_polynomial(2).roots(multiplicities=False)]
            for P in div_2_points:
                if (E_item(X(x=P.x()), Y(x=P.x(),y=P.y())) != P):
                    list_of_3_e_p_structures_sub.append(l_item)
                    flag = True
                    break;
            if flag == True:
                continue
            list_of_3_e_p_structures_max.append(l_item)
        if is_D_e_p_sub == True:
            return list_of_3_e_p_structures_sub
        else:
            return list_of_3_e_p_structures_max
    else:
        raise Exception("Invalid Input")

In [6]:
# Example
d = 3
p = 89
is_D_e_p_sub = True
e = -kronecker(-d, p)

# outputs explicitly the (3, e_{p, 3}^H)-structures
give_explicit_d_e_structure(d, p, is_D_e_p_sub, e)

6

In [8]:
# Returns |D_{3, e}(p))| as given by Chenu and Smith in Corollary 1
# It doesn't matter what e is (although in our case, e = e_{p, 3}^H).

def D_d_e_cardinality(d, p):
    K_im = QuadraticField(-d*p)
    h_K = K_im.class_number()
    if (-d*p % 8) == 1:
        h_K_sub = h_K
    elif (-d*p % 8) == 5:
        h_K_sub = 3*h_K
    else:
        h_K_sub = 0
    return h_K + h_K_sub

In [7]:
# Returns only |D_{d, e_{p, 3}^H}|
# Will rewrite this function to return explicitly the elements of D_{d, e_{p, 3}^H} instead

def hasegawa_3_D_e(d, p):
    if is_prime(p) == False:
        raise Exception("p must be prime!")
    if p <= 11:
        raise Exception("p must be > 11")
    K = GF(p)
    RR.<x> = PolynomialRing(K)
    qns = K.quadratic_nonresidue()
    F.<i> = K.extension(modulus=x^2 - qns)
    D = F(qns)
    D_sqrt = D.sqrt()
    flags = []
    for u in range(0, p):
        try:
            E = EllipticCurve(F, [-3*(5 + 4*u*D_sqrt), 2*(2*u^2*D + 14*u*D_sqrt + 11)])
        except:
            continue
        flag = E.is_supersingular()
        flags.append(flag)
    # includes j-invariant equal to 0
    if (p % 3) == 2:
        return sum(flags)*2 + 2
    elif (p % 3) == 1:
        return sum(flags)*2
    else:
        raise Exception("p is not prime!")

In [ ]:
# Show that there is a bijection between D_{p, 3}^H and D_{d, e_{p, 3}^H}

for p in range(0, 1000):
    if p > 11 and is_prime(p):
        print(f"| {p} | #(D_d_e) = {D_d_e_cardinality(3, p)} | #(hasegawa_3_D_e(3, p)) = {hasegawa_3_D_e(3, p)}")
        assert  D_d_e_cardinality(3, p) == hasegawa_3_D_e(3, p)
    else:
        continue
print("There is a bijection between D_{p, 3}^H and D_{3, e}")